In [0]:
import requests
import json

API_KEY = "339c81a11d3da23b3e35d5796b74b79b"
city = "Seattle"

url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}"

response = requests.get(url)

data = response.json()

print(data)


{'coord': {'lon': -122.3321, 'lat': 47.6062}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 287.44, 'feels_like': 286.96, 'temp_min': 286.21, 'temp_max': 288.55, 'pressure': 1018, 'humidity': 78, 'sea_level': 1018, 'grnd_level': 1008}, 'visibility': 10000, 'wind': {'speed': 2.06, 'deg': 210}, 'clouds': {'all': 100}, 'dt': 1778434380, 'sys': {'type': 2, 'id': 2009669, 'country': 'US', 'sunrise': 1778416705, 'sunset': 1778470386}, 'timezone': -25200, 'id': 5809844, 'name': 'Seattle', 'cod': 200}


In [0]:
with open("/Volumes/workspace/default/Volume/weather_raw.json", "w") as f:
    json.dump(data, f)

In [0]:
raw_df = spark.read.json("/Volumes/workspace/default/Volume/weather_raw.json")
display(raw_df)


base,clouds,cod,coord,dt,id,main,name,sys,timezone,visibility,weather,wind
stations,List(100),200,"List(47.6062, -122.3321)",1778434380,5809844,"List(286.96, 1008, 78, 1018, 1018, 287.44, 288.55, 286.21)",Seattle,"List(US, 2009669, 1778416705, 1778470386, 2)",-25200,10000,"List(List(overcast clouds, 04d, 804, Clouds))","List(210, 2.06)"


In [0]:
raw_df.write.mode("overwrite").saveAsTable("weather_bronze")

In [0]:
from pyspark.sql.functions import col

silver_df = raw_df.select(
    col("name").alias("city"),
    col("main.temp").alias("temperature"),
    col("main.humidity").alias("humidity"),
    col("weather")[0]["description"].alias("weather_condition")
)

display(silver_df)


city,temperature,humidity,weather_condition
Seattle,287.44,78,overcast clouds


In [0]:
silver_df.write.mode("overwrite").saveAsTable("weather_silver")

In [0]:
sensor_df = spark.read.csv(
    "/Volumes/workspace/default/Volume/sensor_data.csv",
    header=True,
    inferSchema=True
)
display(sensor_df)

timestamp,sensor_id,sensor_type,location,value,unit
2024-01-01T10:00:00.000Z,SENS-003,Pressure,Basement,1020.38,hPa
2024-01-01T10:05:00.000Z,SENS-004,Light,Living_Room,508.57,Lux
2024-01-01T10:10:00.000Z,SENS-001,Temperature,Room_A,21.77,Celsius
2024-01-01T10:15:00.000Z,SENS-003,Pressure,Basement,1009.99,hPa
2024-01-01T10:20:00.000Z,SENS-003,Pressure,Basement,998.21,hPa
2024-01-01T10:25:00.000Z,SENS-004,Light,Living_Room,464.01,Lux
2024-01-01T10:30:00.000Z,SENS-001,Temperature,Room_A,21.08,Celsius
2024-01-01T10:35:00.000Z,SENS-001,Temperature,Room_A,24.11,Celsius
2024-01-01T10:40:00.000Z,SENS-003,Pressure,Basement,1016.44,hPa
2024-01-01T10:45:00.000Z,SENS-002,Humidity,Room_A,36.18,Percent


In [0]:
sensor_df.write.mode("overwrite").saveAsTable("sensor_bronze")

In [0]:
clean_sensor_df = sensor_df.dropna()
display(clean_sensor_df)


timestamp,sensor_id,sensor_type,location,value,unit
2024-01-01T10:00:00.000Z,SENS-003,Pressure,Basement,1020.38,hPa
2024-01-01T10:05:00.000Z,SENS-004,Light,Living_Room,508.57,Lux
2024-01-01T10:10:00.000Z,SENS-001,Temperature,Room_A,21.77,Celsius
2024-01-01T10:15:00.000Z,SENS-003,Pressure,Basement,1009.99,hPa
2024-01-01T10:20:00.000Z,SENS-003,Pressure,Basement,998.21,hPa
2024-01-01T10:25:00.000Z,SENS-004,Light,Living_Room,464.01,Lux
2024-01-01T10:30:00.000Z,SENS-001,Temperature,Room_A,21.08,Celsius
2024-01-01T10:35:00.000Z,SENS-001,Temperature,Room_A,24.11,Celsius
2024-01-01T10:40:00.000Z,SENS-003,Pressure,Basement,1016.44,hPa
2024-01-01T10:45:00.000Z,SENS-002,Humidity,Room_A,36.18,Percent


In [0]:
clean_sensor_df.write.mode("overwrite").saveAsTable("sensor_silver")